# CUDA OPTIMISED CODE

In [2]:
!pip install numba-cuda[cu12]

In [10]:
from __future__ import print_function
import numpy as np
from numba import cuda
import math

# CUDA device function for cnd
@cuda.jit(device=True)
def cnd_device(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d)) # Use math.fabs for scalar absolute value
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) * # Use math.exp for scalar exponential
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))

    # Replace np.where with an if/else for scalar operation
    if d > 0:
        return 1.0 - ret_val
    else:
        return ret_val

# CUDA kernel for black_scholes
@cuda.jit
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility,
                         callResult, putResult):
    idx = cuda.grid(1) # Get the global thread ID

    # Ensure we don't go out of bounds for the arrays
    if idx < stockPrice.size:
        # Load scalar values for the current option from device memory
        S = stockPrice[idx]
        X = optionStrike[idx]
        T = optionYears[idx]
        R = Riskfree[idx]
        V = Volatility[idx]

        sqrtT = math.sqrt(T) # Use math.sqrt for scalar square root

        # Handle potential division by zero or log of non-positive numbers
        # This is a simplified check; more robust error handling might be needed
        if V * sqrtT == 0 or S / X <= 0:
            callResult[idx] = 0.0
            putResult[idx] = 0.0
            return

        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT) # Use math.log for scalar logarithm
        d2 = d1 - V * sqrtT

        cndd1 = cnd_device(d1) # Call the device function
        cndd2 = cnd_device(d2)

        expRT = math.exp(- R * T) # Use math.exp for scalar exponential

        # Store results back to device memory
        callResult[idx] = S * cndd1 - X * expRT * cndd2
        putResult[idx] = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# Host-side function to orchestrate data transfer and kernel launch
def black_scholes_optimized(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    # Ensure inputs are NumPy arrays and convert to float64 for precision
    stockPrice = np.asarray(stockPrice, dtype=np.float64)
    optionStrike = np.asarray(optionStrike, dtype=np.float64)
    optionYears = np.asarray(optionYears, dtype=np.float64)
    Riskfree = np.asarray(Riskfree, dtype=np.float64)
    Volatility = np.asarray(Volatility, dtype=np.float64)

    N = stockPrice.size

    # Transfer input arrays from host to device
    d_stockPrice = cuda.to_device(stockPrice)
    d_optionStrike = cuda.to_device(optionStrike)
    d_optionYears = cuda.to_device(optionYears)
    d_Riskfree = cuda.to_device(Riskfree)
    d_Volatility = cuda.to_device(Volatility)

    # Allocate output arrays on the device
    d_callResult = cuda.device_array(N, dtype=np.float64)
    d_putResult = cuda.device_array(N, dtype=np.float64)

    # Configure kernel launch parameters
    threadsperblock = 256 # A common choice for threads per block
    blockspergrid = (N + (threadsperblock - 1)) // threadsperblock # Calculate blocks needed

    # Launch the CUDA kernel
    black_scholes_kernel[blockspergrid, threadsperblock](
        d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
        d_callResult, d_putResult
    )

    # Copy results back from device to host
    callResult = d_callResult.copy_to_host()
    putResult = d_putResult.copy_to_host()

    return callResult, putResult

def black_scholes_gpu_resident(d_S, d_X, d_T, d_R, d_V, d_call, d_put, bpg, tpb=256):
    """
    Zero-copy GPU kernel launcher for batch or simulation workloads.
    Operates entirely on pre-transferred device arrays; no host-device
    transfers occur inside this function. Results are written in-place
    to the caller-supplied d_call and d_put device arrays.
    """
    black_scholes_kernel[bpg, tpb](d_S, d_X, d_T, d_R, d_V, d_call, d_put)



# --- Original CPU functions for verification ---
def cnd_cpu(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S = stockPrice
    X = optionStrike
    T = optionYears
    R = Riskfree
    V = Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1)
    cndd2 = cnd_cpu(d2)

    expRT = np.exp(- R * T)

    callResult = S * cndd1 - X * expRT * cndd2
    putResult = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

    return callResult, putResult

def black_scholes_cpu_resident(S, X, T, R, V, call_out, put_out):
    """
    In-place CPU Black-Scholes pricing.
    Accepts pre-allocated call_out and put_out arrays and writes results
    directly into them, avoiding per-call output allocation overhead.
    Direct equivalent of black_scholes_gpu_resident for fair batch comparison.
    """
    sqrtT = np.sqrt(T)
    d1    = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2    = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1)
    cndd2 = cnd_cpu(d2)
    expRT = np.exp(-R * T)
    np.copyto(call_out, S * cndd1 - X * expRT * cndd2)
    np.copyto(put_out,  X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1))



# --- Example Usage and Verification ---
N_OPTIONS = 10**6 # Number of options to price
# Generate random data, ensuring positive values where required to avoid math domain errors
stockPrice = np.random.rand(N_OPTIONS).astype(np.float64) * 100 + 10
optionStrike = np.random.rand(N_OPTIONS).astype(np.float64) * 100 + 10
optionYears = np.random.rand(N_OPTIONS).astype(np.float64) * 2 + 0.1
Riskfree = np.random.rand(N_OPTIONS).astype(np.float64) * 0.1 + 0.01
Volatility = np.random.rand(N_OPTIONS).astype(np.float64) * 0.5 + 0.01

# Run the optimized Numba-CUDA version
call_opt, put_opt = black_scholes_optimized(stockPrice, optionStrike, optionYears, Riskfree, Volatility)

print("Optimized Call Result (first 5):", call_opt[:5])
print("Optimized Put Result (first 5):", put_opt[:5])

# Run the original CPU version for comparison
call_cpu, put_cpu = black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility)

print("\nCPU Call Result (first 5):", call_cpu[:5])
print("CPU Put Result (first 5):", put_cpu[:5])

# Verify that the results are numerically close
np.testing.assert_allclose(call_opt, call_cpu, rtol=1e-5, atol=1e-8)
np.testing.assert_allclose(put_opt, put_cpu, rtol=1e-5, atol=1e-8)
print("\nVerification successful: Results from Numba-CUDA match the original CPU version.")

Optimized Call Result (first 5): [7.35657199e-02 4.38098617e+01 5.74863588e+01 3.96806133e+00
 2.18490892e-55]
Optimized Put Result (first 5): [1.86186196e+01 0.00000000e+00 3.50808787e-15 3.91017032e+01
 8.14298354e+01]

CPU Call Result (first 5): [7.35657199e-02 4.38098617e+01 5.74863588e+01 3.96806133e+00
 2.18490892e-55]
CPU Put Result (first 5): [1.86186196e+01 0.00000000e+00 3.50808787e-15 3.91017032e+01
 8.14298354e+01]

Verification successful: Results from Numba-CUDA match the original CPU version.


# Keeping Data in GPU Memory

In [11]:
import warnings
import time
import timeit
import statistics

import numpy as np
from numba import cuda
from numba.core.errors import NumbaPerformanceWarning
import math

warnings.filterwarnings("ignore", category=NumbaPerformanceWarning)


# ─────────────────────────────────────────────────────────────────────────────
# INPUT FACTORIES
# ─────────────────────────────────────────────────────────────────────────────

def make_inputs(n):
    """Generate pageable host arrays with randomised option parameters."""
    return dict(
        stockPrice   = np.random.uniform(10,   150, n).astype(np.float64),
        optionStrike = np.random.uniform(10,   150, n).astype(np.float64),
        optionYears  = np.random.uniform(0.1,  2.0, n).astype(np.float64),
        Riskfree     = np.random.uniform(0.01, 0.1, n).astype(np.float64),
        Volatility   = np.random.uniform(0.01, 0.5, n).astype(np.float64),
    )


def make_inputs_pinned(n):
    """
    Generate page-locked (pinned) host arrays with randomised option parameters.
    Pinned memory enables direct DMA transfers, eliminating the CPU-side
    staging copy and saturating available PCIe bandwidth.
    """
    def pinned_uniform(lo, hi, size):
        arr = cuda.pinned_array(size, dtype=np.float64)
        arr[:] = np.random.uniform(lo, hi, size)
        return arr

    return dict(
        stockPrice   = pinned_uniform(10,   150, n),
        optionStrike = pinned_uniform(10,   150, n),
        optionYears  = pinned_uniform(0.1,  2.0, n),
        Riskfree     = pinned_uniform(0.01, 0.1, n),
        Volatility   = pinned_uniform(0.01, 0.5, n),
    )


def make_device_inputs(inp):
    """
    Transfer host arrays to the device and pre-allocate output buffers.
    Reusing these buffers across repeated kernel launches avoids
    per-call device allocation overhead.
    """
    N = inp["stockPrice"].size
    return {
        "d_S"    : cuda.to_device(inp["stockPrice"]),
        "d_X"    : cuda.to_device(inp["optionStrike"]),
        "d_T"    : cuda.to_device(inp["optionYears"]),
        "d_R"    : cuda.to_device(inp["Riskfree"]),
        "d_V"    : cuda.to_device(inp["Volatility"]),
        "d_call" : cuda.device_array(N, dtype=np.float64),
        "d_put"  : cuda.device_array(N, dtype=np.float64),
        "bpg"    : (N + THREADSPERBLOCK - 1) // THREADSPERBLOCK,
    }


def make_device_inputs_pinned(inp_pinned):
    """
    Transfer pinned host arrays to the device and pre-allocate output buffers.
    Combining pinned source memory with pre-allocated outputs minimises
    both transfer latency and device-side allocation overhead.
    """
    N = inp_pinned["stockPrice"].size
    return {
        "d_S"    : cuda.to_device(inp_pinned["stockPrice"]),
        "d_X"    : cuda.to_device(inp_pinned["optionStrike"]),
        "d_T"    : cuda.to_device(inp_pinned["optionYears"]),
        "d_R"    : cuda.to_device(inp_pinned["Riskfree"]),
        "d_V"    : cuda.to_device(inp_pinned["Volatility"]),
        "d_call" : cuda.device_array(N, dtype=np.float64),
        "d_put"  : cuda.device_array(N, dtype=np.float64),
        "bpg"    : (N + THREADSPERBLOCK - 1) // THREADSPERBLOCK,
    }


# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

if not cuda.is_available():
    raise RuntimeError("No CUDA-capable GPU detected. This benchmark requires a GPU.")

gpu = cuda.get_current_device()
print(f"GPU                : {gpu.name.decode()}")
print(f"Compute Capability : {gpu.compute_capability}")
print(f"Max Threads/Block  : {gpu.MAX_THREADS_PER_BLOCK}")
print(f"Multiprocessors    : {gpu.MULTIPROCESSOR_COUNT}\n")

REPEATS         = 10
NUMBER          = 5
THREADSPERBLOCK = 256
SIZES           = [100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]
BATCH_SIZE      = 100

np.random.seed(42)

# One-time JIT compilation — excluded from all timed measurements
print("Compiling CUDA kernel ...")
black_scholes_optimized(**make_inputs(1024))
cuda.synchronize()
print("Compilation complete.\n")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1 — End-to-End Latency: Pageable Memory
# Measures the full pipeline (H2D + kernel + D2H) using standard
# pageable host allocations. Establishes the unoptimised baseline.
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print("PHASE 1 — End-to-End Latency: Pageable Memory")
print(f"  {REPEATS} repeats × {NUMBER} calls each\n")
print(f"{'Size':>10} {'Min(ms)':>10} {'Max(ms)':>10} {'Mean(ms)':>10} "
      f"{'Std(ms)':>10} {'Tput M/s':>10}")
print("─" * 62)

e2e_results = []
for n in SIZES:
    inp = make_inputs(n)
    black_scholes_optimized(**inp); cuda.synchronize()

    raw = timeit.repeat(
        stmt    = "black_scholes_optimized(**inp); cuda.synchronize()",
        setup   = "pass",
        globals = {"black_scholes_optimized": black_scholes_optimized,
                   "cuda": cuda, "inp": inp},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    per_call_ms = [t / NUMBER * 1000 for t in raw]
    mn = min(per_call_ms); mx = max(per_call_ms)
    mean = statistics.mean(per_call_ms); std = statistics.stdev(per_call_ms)
    tput = n / (mean / 1000) / 1e6
    e2e_results.append(dict(size=n, min=mn, max=mx, mean=mean, std=std, throughput=tput))
    print(f"{n:>10,} {mn:>10.3f} {mx:>10.3f} {mean:>10.3f} {std:>10.4f} {tput:>10.2f}")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1b — End-to-End Latency: Pinned Memory
# Identical pipeline to Phase 1 with page-locked host allocations.
# Direct DMA transfers bypass the CPU staging copy, reducing PCIe latency.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 1b — End-to-End Latency: Pinned Memory")
print(f"  {REPEATS} repeats × {NUMBER} calls each\n")
print(f"{'Size':>10} {'Min(ms)':>10} {'Max(ms)':>10} {'Mean(ms)':>10} "
      f"{'Std(ms)':>10} {'Tput M/s':>10} {'vs 1a':>10}")
print("─" * 74)

e2e_pinned_results = []
for idx, n in enumerate(SIZES):
    inp_p = make_inputs_pinned(n)
    black_scholes_optimized(**inp_p); cuda.synchronize()

    raw = timeit.repeat(
        stmt    = "black_scholes_optimized(**inp_p); cuda.synchronize()",
        setup   = "pass",
        globals = {"black_scholes_optimized": black_scholes_optimized,
                   "cuda": cuda, "inp_p": inp_p},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    per_call_ms = [t / NUMBER * 1000 for t in raw]
    mn = min(per_call_ms); mx = max(per_call_ms)
    mean = statistics.mean(per_call_ms); std = statistics.stdev(per_call_ms)
    tput = n / (mean / 1000) / 1e6
    improvement = e2e_results[idx]["mean"] / mean
    e2e_pinned_results.append(dict(size=n, min=mn, max=mx, mean=mean, std=std, throughput=tput))
    print(f"{n:>10,} {mn:>10.3f} {mx:>10.3f} {mean:>10.3f} {std:>10.4f} "
          f"{tput:>10.2f} {improvement:>9.2f}x")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2 — Kernel-Only Latency
# Data pre-transferred to the device before timing begins.
# Isolates pure GPU compute time from PCIe transfer overhead.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 2 — Kernel-Only Latency (device-resident data)")
print(f"  {REPEATS} repeats × {NUMBER} calls each\n")
print(f"{'Size':>10} {'Min(ms)':>10} {'Max(ms)':>10} {'Mean(ms)':>10} "
      f"{'Std(ms)':>10} {'Tput M/s':>10}")
print("─" * 62)

kernel_results = []
for n in SIZES:
    inp = make_inputs(n)
    d   = make_device_inputs(inp)
    bpg = d["bpg"]

    def run_kernel():
        black_scholes_kernel[bpg, THREADSPERBLOCK](
            d["d_S"], d["d_X"], d["d_T"], d["d_R"], d["d_V"],
            d["d_call"], d["d_put"])
        cuda.synchronize()

    run_kernel()

    raw = timeit.repeat(
        stmt    = "run_kernel()",
        setup   = "pass",
        globals = {"run_kernel": run_kernel},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    per_call_ms = [t / NUMBER * 1000 for t in raw]
    mn = min(per_call_ms); mx = max(per_call_ms)
    mean = statistics.mean(per_call_ms); std = statistics.stdev(per_call_ms)
    tput = n / (mean / 1000) / 1e6
    kernel_results.append(dict(size=n, min=mn, max=mx, mean=mean, std=std, throughput=tput))
    print(f"{n:>10,} {mn:>10.3f} {mx:>10.3f} {mean:>10.3f} {std:>10.4f} {tput:>10.2f}")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3 — PCIe Transfer Breakdown (H2D = Host-to-Device) (D2H = Device-to-Host)
# H2D and D2H transfers measured independently to quantify what fraction
# of end-to-end latency is attributable to data movement.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 3 — PCIe Transfer Breakdown (H2D and D2H)\n")
print(f"{'Size':>10} {'H2D(ms)':>12} {'D2H(ms)':>12} {'Transfer%(e2e)':>16}")
print("─" * 54)

for idx, n in enumerate(SIZES):
    inp = make_inputs(n)
    _ = cuda.to_device(inp["stockPrice"]); cuda.synchronize()

    raw_h2d = timeit.repeat(
        stmt = (
            "cuda.to_device(inp['stockPrice']);"
            "cuda.to_device(inp['optionStrike']);"
            "cuda.to_device(inp['optionYears']);"
            "cuda.to_device(inp['Riskfree']);"
            "cuda.to_device(inp['Volatility']);"
            "cuda.synchronize()"
        ),
        setup   = "pass",
        globals = {"cuda": cuda, "inp": inp},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    h2d_ms = min(raw_h2d) / NUMBER * 1000

    d_call = cuda.device_array(n, dtype=np.float64)
    d_put  = cuda.device_array(n, dtype=np.float64)
    d_call.copy_to_host(); cuda.synchronize()

    raw_d2h = timeit.repeat(
        stmt    = "d_call.copy_to_host(); d_put.copy_to_host(); cuda.synchronize()",
        setup   = "pass",
        globals = {"cuda": cuda, "d_call": d_call, "d_put": d_put},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    d2h_ms = min(raw_d2h) / NUMBER * 1000

    transfer_pct = (h2d_ms + d2h_ms) / e2e_results[idx]["min"] * 100
    print(f"{n:>10,} {h2d_ms:>12.3f} {d2h_ms:>12.3f} {transfer_pct:>16.1f}%")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4 — GPU vs CPU Speedup Summary
# Compares pageable, pinned, and kernel-only GPU paths against the
# vectorised NumPy CPU baseline for each array size.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 4 — GPU vs NumPy CPU Speedup\n")
print(f"{'Size':>10} {'CPU(ms)':>10} {'E2E-Page':>10} {'E2E-Pin':>9} "
      f"{'Kern(ms)':>10} {'Spd-Page':>10} {'Spd-Pin':>9} {'Spd-Kern':>10}")
print("─" * 84)

for idx, n in enumerate(SIZES):
    inp = make_inputs(n)
    black_scholes_cpu(**inp)

    cpu_raw = timeit.repeat(
        stmt    = "black_scholes_cpu(**inp)",
        setup   = "pass",
        globals = {"black_scholes_cpu": black_scholes_cpu, "inp": inp},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    cpu_ms = min(t / NUMBER * 1000 for t in cpu_raw)

    gpu_e2e_page = e2e_results[idx]["min"]
    gpu_e2e_pin  = e2e_pinned_results[idx]["min"]
    gpu_kern_ms  = kernel_results[idx]["min"]

    print(f"{n:>10,} {cpu_ms:>10.3f} {gpu_e2e_page:>10.3f} {gpu_e2e_pin:>9.3f} "
          f"{gpu_kern_ms:>10.3f} {cpu_ms/gpu_e2e_page:>9.2f}x "
          f"{cpu_ms/gpu_e2e_pin:>8.2f}x {cpu_ms/gpu_kern_ms:>9.2f}x")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 5 — Numerical Correctness Verification
# Confirms GPU and CPU results agree within floating-point tolerances.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 5 — Numerical Correctness Verification\n")
inp = make_inputs(1_000_000)
call_gpu, put_gpu = black_scholes_optimized(**inp)
call_cpu, put_cpu = black_scholes_cpu(**inp)
np.testing.assert_allclose(call_gpu, call_cpu, rtol=1e-5, atol=1e-8)
np.testing.assert_allclose(put_gpu,  put_cpu,  rtol=1e-5, atol=1e-8)
print("✓ GPU and CPU results agree within tolerances (rtol=1e-5, atol=1e-8)")
print(f"  Max absolute error — Call: {np.max(np.abs(call_gpu - call_cpu)):.2e}"
      f"  |  Put: {np.max(np.abs(put_gpu - put_cpu)):.2e}")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 6 — GPU-Resident vs CPU-Resident Batch Pricing
# Both sides pre-allocate output buffers once and reuse them across the
# entire batch. The GPU additionally amortises the one-time PCIe transfer
# cost over BATCH_SIZE kernel runs.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print(f"PHASE 6 — Resident Batch Pricing  (batch size = {BATCH_SIZE})")
print("  GPU: transfer once → kernel × N → copy back once")
print("  CPU: pre-allocate outputs once → compute × N  (no alloc per call)\n")
print(f"{'Size':>10} {'CPU-Res(ms)':>13} {'GPU-Res(ms)':>13} "
      f"{'Eff.Spd':>9} {'vs Phase1':>10}")
print("─" * 62)

for idx, n in enumerate(SIZES):

    # ── CPU-resident setup: pre-allocate inputs and output buffers ────
    inp_cpu  = make_inputs(n)
    S, X     = inp_cpu["stockPrice"],   inp_cpu["optionStrike"]
    T, R, V  = inp_cpu["optionYears"],  inp_cpu["Riskfree"], inp_cpu["Volatility"]
    call_out = np.empty(n, dtype=np.float64)   # pre-allocated output — reused every call
    put_out  = np.empty(n, dtype=np.float64)   # pre-allocated output — reused every call

    # CPU warm-up
    black_scholes_cpu_resident(S, X, T, R, V, call_out, put_out)

    t0 = time.perf_counter()
    for _ in range(BATCH_SIZE):
        black_scholes_cpu_resident(S, X, T, R, V, call_out, put_out)
    t_cpu_res = (time.perf_counter() - t0) * 1000

    # ── GPU-resident setup: pinned transfer once, pre-allocate outputs ─
    inp_p  = make_inputs_pinned(n)
    d      = make_device_inputs_pinned(inp_p)
    bpg    = d["bpg"]

    # GPU warm-up
    black_scholes_gpu_resident(
        d["d_S"], d["d_X"], d["d_T"], d["d_R"], d["d_V"],
        d["d_call"], d["d_put"], bpg)
    cuda.synchronize()

    t0     = time.perf_counter()
    d_S    = cuda.to_device(inp_p["stockPrice"])
    d_X    = cuda.to_device(inp_p["optionStrike"])
    d_T    = cuda.to_device(inp_p["optionYears"])
    d_R    = cuda.to_device(inp_p["Riskfree"])
    d_V    = cuda.to_device(inp_p["Volatility"])
    d_call = cuda.device_array(n, dtype=np.float64)
    d_put  = cuda.device_array(n, dtype=np.float64)

    for _ in range(BATCH_SIZE):
        black_scholes_gpu_resident(d_S, d_X, d_T, d_R, d_V, d_call, d_put, bpg)
    cuda.synchronize()

    d_call.copy_to_host()
    d_put.copy_to_host()
    t_gpu_res = (time.perf_counter() - t0) * 1000

    eff_speedup = t_cpu_res / t_gpu_res
    vs_phase1   = e2e_results[idx]["mean"] * BATCH_SIZE / t_gpu_res

    print(f"{n:>10,} {t_cpu_res:>13.1f} {t_gpu_res:>13.1f} "
          f"{eff_speedup:>8.1f}x {vs_phase1:>9.1f}x")

print(f"\n  Effective speedup converges toward the kernel-only limit as batch size grows.")
print(f"  'vs Phase1' column shows improvement over {BATCH_SIZE} × individual calls.")


GPU                : Tesla T4
Compute Capability : (7, 5)
Max Threads/Block  : 1024
Multiprocessors    : 40

Compiling CUDA kernel ...
Compilation complete.

PHASE 1 — End-to-End Latency: Pageable Memory
  10 repeats × 5 calls each

      Size    Min(ms)    Max(ms)   Mean(ms)    Std(ms)   Tput M/s
──────────────────────────────────────────────────────────────
       100      1.307      1.613      1.372     0.0871       0.07
     1,000      1.345      1.753      1.489     0.1341       0.67
    10,000      1.502      1.780      1.584     0.0753       6.31
   100,000      4.066      4.317      4.162     0.0801      24.03
 1,000,000     18.338     21.324     19.974     1.0550      50.07
10,000,000    169.691    175.773    172.790     1.8587      57.87

PHASE 1b — End-to-End Latency: Pinned Memory
  10 repeats × 5 calls each

      Size    Min(ms)    Max(ms)   Mean(ms)    Std(ms)   Tput M/s      vs 1a
──────────────────────────────────────────────────────────────────────────
       100     

Here is a complete breakdown of every phase in above results.

***
## GPU Hardware: Tesla T4
The T4 has 40 SMs with Compute Capability 7.5 (Turing), supporting up to 1024 threads/block . With `THREADSPERBLOCK=256`, you need at least 80 active blocks (`2 × 40 SMs`) to fully saturate the GPU - that requires `n ≥ 20,480` elements, which explains the poor throughput at n=100 and n=1,000.

***
## Phase 1 vs Phase 1b - Pinned Memory Impact
Pinned memory delivers **no meaningful gain below n=10,000** (0.97-1.01×) because the fixed CUDA driver overhead (~1.3 ms) completely overwhelms the transfer itself. The gains only emerge where data volume becomes large enough for the DMA bandwidth difference to matter - **1.25× at n=100k, 1.58× at n=1M, 1.42× at n=10M**. The slight dip to 1.42× at 10M (vs 1.58× at 1M) is because at very large sizes, PCIe bandwidth saturation starts affecting both pageable and pinned transfers, narrowing the gap.

***
## Phase 2 - Kernel Ceiling
The kernel-only throughput plateaus at **692 M options/sec at n=10M** (vs 668 M/s at n=1M), confirming the GPU is fully saturated and memory-bandwidth-bound inside the device . There is nothing left to squeeze from the kernel itself - the 87× speedup at n=10M is the hard ceiling this GPU can deliver.

***
## Phase 3 - PCIe Is the Real Bottleneck
Transfer accounts for **75-90% of every single end-to-end call**. One important detail: D2H at n=10M (59ms) is nearly as expensive as H2D (93ms) despite reading only 2 arrays versus writing 5. This is because the T4 in a cloud VM uses pageable host memory for the output, requiring an extra CPU memcpy from a pinned staging buffer before results reach your NumPy array.

***
## Phase 4 - Full Speedup Summary
Three distinct performance regimes are visible:

| Size range | Winner | Why |
|---|---|---|
| n ≤ 10,000 | **CPU** | PCIe fixed overhead (~1.4 ms) exceeds total CPU compute time |
| n = 100,000 | **GPU wins** | Pinned: 1.85×, Pageable: 1.48× - compute starts paying off |
| n ≥ 1,000,000 | **GPU dominant** | Pinned reaches 10.41×, kernel-only 87× |

The break-even point shifted from ~100k (pageable) to somewhere between 10k and 100k with pinned memory - pinned memory moved the crossover point to a smaller problem size.

***
## Phase 6 - The Most Important Result
This is where everything comes together. At **n=10M with batch=100**, the GPU-resident approach processes 1 billion options in 1,653 ms while CPU-resident takes 129,130 ms - a **78.1× real-world speedup**, compared to only 7.32× in the naive single-call approach. The "vs Phase1" column tells you how much you left on the table with the naive approach: at n=10M, batch pricing is **10.5× faster than making 100 individual calls**.

The progression toward the 87× kernel ceiling as array size grows is clear:

| Size | Batch Speedup | % of kernel ceiling (87×) |
|------|:------------:|:-------------------------:|
| 100 | 0.9× | 1% |
| 1,000 | 1.5× | 2% |
| 10,000 | 6.2× | 7% |
| 100,000 | 15.5× | 18% |
| 1,000,000 | 32.1× | 37% |
| 10,000,000 | **78.1×** | **90%** |

The batch-resident strategy at n=10M recovers **90% of the theoretical kernel-ceiling speedup** in a real end-to-end scenario - the remaining 10% gap is the amortised PCIe cost of the one-time transfer divided across 100 runs.